In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

df = pd.read_csv("data_raw.csv")


In [23]:
print("Dataset shape:", df.shape)

Dataset shape: (4000, 15)


In [24]:
df.head()

,name,rating,genre,year,released,score,votes,director,writer,star,country,budget,gross,company,"runtime,,"
0,The Shining,R,Drama,1980,"June 13, 1980 (United States)",8.4,927000,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,19000000,46998772.0,Warner Bros.,"146.0,"
1,The Blue Lagoon,R,Adventure,1980,"July 2, 1980 (United States)",5.8,65000,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,4500000,58853106.0,Columbia Pictures,"104.0,"
2,Star Wars: Episode V - The Empire Strikes Back,PG,Action,1980,"June 20, 1980 (United States)",8.7,1200000,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,18000000,538375067.0,Lucasfilm,"124.0,"
3,Airplane!,PG,Comedy,1980,"July 2, 1980 (United States)",7.7,221000,Jim Abrahams,Jim Abrahams,Robert Hays,United States,3500000,83453539.0,Paramount Pictures,"88.0,"
4,Caddyshack,R,Comedy,1980,"July 25, 1980 (United States)",7.3,108000,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,6000000,39846344.0,Orion Pictures,"98.0,"


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       4000 non-null   object 
 1   rating     3960 non-null   object 
 2   genre      4000 non-null   object 
 3   year       4000 non-null   int64  
 4   released   4000 non-null   object 
 5   score      4000 non-null   float64
 6   votes      4000 non-null   int64  
 7   director   4000 non-null   object 
 8   writer     3999 non-null   object 
 9   star       3999 non-null   object 
 10  country    4000 non-null   object 
 11  budget     4000 non-null   int64  
 12  gross      3831 non-null   float64
 13  company    3990 non-null   object 
 14  runtime,,  4000 non-null   object 
dtypes: float64(2), int64(3), object(10)
memory usage: 468.9+ KB


# Data Cleaning 

solving year vs released redunduncy

In [26]:
df['release_date'] = df['released'].str.split('(').str[0].str.strip()
df['release_country'] = df['released'].str.split('(').str[1].str.replace(')', '').str.strip()

In [27]:
df.drop(columns=['released' , 'year'], inplace=True)

In [28]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

Filling null values

In [29]:
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 name                 0
rating              40
genre                0
score                0
votes                0
director             0
writer               1
star                 1
country              0
budget               0
gross              169
company             10
runtime,,            0
release_date        53
release_country      0
dtype: int64


In [30]:
for col in ['rating', 'writer', 'star' , 'company' , 'release_country' , 'release_date']:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Filled missing values in '{col}' with mode: {mode_val}")

Filled missing values in 'rating' with mode: R
Filled missing values in 'writer' with mode: Stephen King
Filled missing values in 'star' with mode: Robert De Niro
Filled missing values in 'company' with mode: Paramount Pictures
Filled missing values in 'release_country' with mode: United States
Filled missing values in 'release_date' with mode: 1986-02-14 00:00:00


In [31]:
median_gross = df['gross'].median()
df['gross'] = df['gross'].fillna(median_gross)
print(f"Filled missing values in 'gross' with median: {median_gross}")

Filled missing values in 'gross' with median: 11838218.0


Filling or dropping duplicates 


In [32]:
df.duplicated().sum()

np.int64(0)

# Feature Transformation

encoding categorical coloumns using label encoder

In [33]:
categorical_cols = ['rating', 'genre', 'country', 'director', 'writer', 'star', 'company' , 'release_country']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le 
    print(f"Encoded column '{col}'")

Encoded column 'rating'
Encoded column 'genre'
Encoded column 'country'
Encoded column 'director'
Encoded column 'writer'
Encoded column 'star'
Encoded column 'company'
Encoded column 'release_country'


converting runtime,, into float , trimimg commas renaming it to runtime

In [34]:
df['runtime'] = df['runtime,,'].str.replace(',', '').astype(float)
df = df.drop(columns=['runtime,,'])  

Scaling numerical coloumns 

In [35]:
numerical_cols = ['budget', 'gross', 'score', 'votes', 'runtime']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
print("\nScaled numerical columns:", numerical_cols)



Scaled numerical columns: ['budget', 'gross', 'score', 'votes', 'runtime']


In [36]:
df.head()

,name,rating,genre,score,votes,director,writer,star,country,budget,gross,company,release_date,release_country,runtime
0,The Shining,6,6,2.127016,6.111868,1464,2349,616,39,0.158898,0.072015,1273,1980-06-13,36,2.181487
1,The Blue Lagoon,6,1,-0.534249,0.073591,1282,956,170,40,-0.486017,0.202562,393,1980-07-02,36,-0.092686
2,Star Wars: Episode V - The Empire Strikes Back,4,0,2.434085,8.024223,629,1504,1033,40,0.114421,5.483336,870,1980-06-20,36,0.990254
3,Airplane!,4,4,1.410521,1.166365,736,1182,1344,40,-0.530494,0.473476,1017,1980-07-02,36,-0.959038
4,Caddyshack,6,4,1.001096,0.374804,588,291,221,40,-0.419302,-0.006752,999,1980-07-25,36,-0.417568


# Dimensionality Reduction


 select subset of columns and dropping irrelevent ones

In [37]:
selected_cols = ['rating', 'genre', 'score', 'votes', 'budget', 'gross', 'runtime' , 'release_country', 'release_date']
df_reduced = df[selected_cols].copy()

In [38]:
df_reduced['score_bin'] = pd.qcut(df_reduced['score'], q=3, labels=['Low','Medium','High'])

In [39]:
score_bin_mapping = {"Low": 0, "Medium": 1, "High": 2}
df_reduced['score_bin'] = df_reduced['score_bin'].map(score_bin_mapping)

In [40]:
df_reduced

,rating,genre,score,votes,budget,gross,runtime,release_country,release_date,score_bin
0,6,6,2.127016,6.111868,0.158898,0.072015,2.181487,36,1980-06-13,2
1,6,1,-0.534249,0.073591,-0.486017,0.202562,-0.092686,36,1980-07-02,0
2,4,0,2.434085,8.024223,0.114421,5.483336,0.990254,36,1980-06-20,2
3,4,4,1.410521,1.166365,-0.530494,0.473476,-0.959038,36,1980-07-02,2
4,6,4,1.001096,0.374804,-0.419302,-0.006752,-0.417568,36,1980-07-25,2
...,...,...,...,...,...,...,...,...,...,...
3995,6,4,-0.227180,-0.206608,-0.107963,-0.267477,-0.688303,36,2002-02-01,1
3996,6,4,0.079889,-0.255643,-0.686163,-0.428222,-0.525862,9,2001-09-12,1
3997,6,4,0.079889,-0.255643,-0.463779,-0.385207,-0.580009,36,2001-08-31,1
3998,6,4,-0.227180,-0.178588,0.114421,-0.296032,-0.688303,36,2001-04-27,1


In [41]:
df_reduced.to_csv("data_preprocessed.csv", index=False)
print("Saved preprocessed data as data_preprocessed.csv")

Saved preprocessed data as data_preprocessed.csv
